# web_agent — Kaggle driver (hybrid)

Stable core (`labels`, `config`, `dataset`, `model`, `loss`, `metrics`) lives in the
`web_agent` package, cloned from GitHub. Orchestration + analysis (training loop,
eval, plots, error inspection, result images) lives **here in the notebook** so you
can watch every step.

Workflow: edit package in IDE → push to GitHub `Code` → **Pull** here → re-run.

In [2]:
# 1. Clone the package from GitHub and install it (re-run safe)
import os
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
if not os.path.isdir("/kaggle/working/webagent"):
    !git clone -b Code {REPO} /kaggle/working/webagent
else:
    !cd /kaggle/working/webagent && git pull --ff-only
%cd /kaggle/working/webagent
!pip install -e . -q
print("package ready")

Cloning into '/kaggle/working/webagent'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 71 (delta 1), reused 71 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 55.87 KiB | 2.23 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/webagent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.4 MB/s eta 0:00:00:00:0100:01
  Building editable for web_agent (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages th

In [3]:
# 2. Confirm GPU + dataset path, then verify the dataset (Phase 1)
import os
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))

# Adjust if the mount name differs from this:
DATA_PATH = "/kaggle/input/datasets/kiyasmahmud/thesisdata/FinalData"
assert os.path.isdir(DATA_PATH), f"fix DATA_PATH; not found: {DATA_PATH}"

!python -m web_agent.data.verify_dataset --data {DATA_PATH} --strict

GPU 0: Tesla T4 (UUID: GPU-f9a6533f-a081-1d48-9e17-f1d9c36dc681)
GPU 1: Tesla T4 (UUID: GPU-eb22e895-08da-5994-d1b0-e981b433188a)
input dirs: ['datasets']
/usr/bin/python3: Error while finding module specification for 'web_agent.data.verify_dataset' (ModuleNotFoundError: No module named 'web_agent.data')


In [4]:
# 3. Phase 2 — build a DataLoader and pull ONE batch (Y1: SigLIP + RoBERTa)
from transformers import AutoProcessor, AutoTokenizer
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.dataloader import load_split, build_dataloader

set_seed(42)
cfg = load_config("configs/backbones/y1_siglip_roberta.yaml")
cfg["data"]["root"] = DATA_PATH            # point at the Kaggle mount
cfg["data"]["num_workers"] = 2

processor = AutoProcessor.from_pretrained(cfg["backbone"]["vision_encoder"])
tokenizer = AutoTokenizer.from_pretrained(cfg["backbone"]["text_encoder"])

train = load_split(cfg, "train")
loader = build_dataloader(cfg, mode="full_labels", records=train,
                          processor=processor, tokenizer=tokenizer,
                          limit=cfg["stages"]["smoke"], batch_size=8)

batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    print(f"  {k:18} {tuple(v.shape)}  {v.dtype}")

ModuleNotFoundError: No module named 'web_agent'

In [ ]:
# 4. Visual sanity-check — show 4 raw screenshots with their decoded labels
import matplotlib.pyplot as plt
from PIL import Image
from web_agent.labels import (
    EXECUTION_OUTCOME_INV, FAILURE_TYPE_INV, ACTION_TYPE_INV, RECOVERY_STRATEGY_INV,
)

rows = loader.dataset.records[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, rec in zip(axes, rows):
    ax.imshow(Image.open(f"{DATA_PATH}/{rec['state_before']}").convert("RGB"))
    ax.axis("off")
    ax.set_title(
        f"{rec['execution_outcome']} / {rec['failure_type']}\n"
        f"act={rec['action_type']}  rec={rec['recovery_strategy']}\n"
        f"conf={rec['agent_confidence_before']:.2f}  bbox={'Y' if rec['action_target_bbox'] else 'N'}",
        fontsize=9,
    )
plt.tight_layout(); plt.show()

# Confirm encoded labels match the strings above
print("encoded label_outcome :", batch["label_outcome"][:4].tolist())
print("encoded label_action  :", batch["label_action"][:4].tolist())
print("bbox_mask             :", batch["bbox_mask"][:4].squeeze(-1).tolist())